In [10]:
from pathlib import Path
import math
import random
import shutil

import pandas as pd
import pyrosetta
from pyrosetta import pose_from_pdb
from pyrosetta.rosetta.core.id import AtomID
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.core.pack.task import TaskFactory
from pyrosetta.rosetta.core.pack.task.operation import (
IncludeCurrent,InitializeFromCommandline,NoRepackDisulfides,OperateOnResidueSubset,PreventRepackingRLT,RestrictToRepackingRLT,)
from pyrosetta.rosetta.core.scoring import atom_pair_constraint
from pyrosetta.rosetta.core.scoring.constraints import AtomPairConstraint
from pyrosetta.rosetta.core.scoring.func import HarmonicFunc
from pyrosetta.rosetta.core.select.residue_selector import (NeighborhoodResidueSelector,ResidueIndexSelector,)
from pyrosetta.rosetta.protocols.backrub import BackrubMover
from pyrosetta.rosetta.protocols.minimization_packing import PackRotamersMover
from pyrosetta.rosetta.protocols.relax import FastRelax

pyrosetta.init("-mute all")

PDB_PATH = Path("/mnt/c/Users/kevin/Downloads/RYR2_PDB_STRUCTURES/PDB_STRUCTURES/5goa_A165D.pdb") #CHANGE PATH
OUTDIR = Path("/mnt/c/Users/kevin/Downloads/A165D open fastrelax") #CHANGE OUTDIR
OUTDIR.mkdir(parents=True, exist_ok=True)

CHAIN = "A"
PDB_FLEX_START = 160 #define loop parameters
PDB_FLEX_END = 180

NSTRUCT = 20 # number of trials
BACKRUB_TRIALS = 250 #backrub does 250 independent moves per model but not all are accepted by monte carlo acceptance

DIST_CST_CUTOFF = 20.0 #can repack for residues 20 A away from 160-180
DIST_CST_SD = 6.0 #can repack past 6 A movement but is not favorable
DIST_CST_WEIGHT = 0.05 #make constraints matter less with smaller number
KT = 10.0 #accept energy, larger = higher acceprtance

FASTRELAX_REPEATS = 1 #one fastrelx run
FASTRELAX_MAX_ITER = 200 #only 200 allowed fastrelax moves within the protocol


pose0 = pose_from_pdb(str(PDB_PATH))
pdb_info = pose0.pdb_info()

scorefxn = pyrosetta.create_score_function("ref2015") #default all-atom rosetta score function
original_score = scorefxn(pose0)
print(f"Original score: {original_score:.3f} REU")

FLEX_START = pdb_info.pdb2pose(CHAIN, PDB_FLEX_START) #convert to rosetta numbering pose 
FLEX_END = pdb_info.pdb2pose(CHAIN, PDB_FLEX_END) 

cst_scorefxn = scorefxn.clone()  #new score function for constrained
cst_scorefxn.set_weight(atom_pair_constraint, DIST_CST_WEIGHT)

loop_residues = ",".join(str(residue) for residue in range(FLEX_START, FLEX_END + 1))
loop_selector = ResidueIndexSelector(loop_residues)

repack_selector = NeighborhoodResidueSelector() #allows distance constraints to select and limit neighboring residues
repack_selector.set_focus_selector(loop_selector)
repack_selector.set_distance(DIST_CST_CUTOFF)
repack_selector.set_include_focus_in_subset(True)

repack_subset = repack_selector.apply(pose0)


def add_distance_constraints(pose, reference_pose, start, end):
    """Constrain loop-neighborhood CA distances to their starting values."""
    added = 0
    seen_pairs = set()
    neighborhood = repack_selector.apply(reference_pose)

    for r1 in range(start, end + 1):
        residue1 = reference_pose.residue(r1)

        for r2 in range(1, reference_pose.total_residue() + 1):
            residue2 = reference_pose.residue(r2)
            pair = tuple(sorted((r1, r2)))

            if r1 == r2 or pair in seen_pairs:
                continue
            if not neighborhood[r2]:
                continue

            xyz1 = residue1.xyz("CA")  #3d coordinate distance calculation sqrt(x1-x2)^2 + (y1-y2)^2 + (z1-z2)^2
            xyz2 = residue2.xyz("CA")
            distance = math.sqrt((xyz1.x - xyz2.x) ** 2 + (xyz1.y - xyz2.y) ** 2 + (xyz1.z - xyz2.z) ** 2)
            if distance > DIST_CST_CUTOFF:
                continue

            atom1 = AtomID(residue1.atom_index("CA"), r1)
            atom2 = AtomID(residue2.atom_index("CA"), r2)
            constraint = AtomPairConstraint(atom1, atom2, HarmonicFunc(distance, DIST_CST_SD))
            pose.add_constraint(constraint)
            seen_pairs.add(pair)
            added += 1

    return added


# Backrub perturbs only the selected chain-A backbone region.
backrub_movemap = MoveMap()
backrub_movemap.set_bb_true_range(FLEX_START, FLEX_END)
backrub = BackrubMover()
backrub.set_movemap(backrub_movemap)
backrub.set_min_atoms(3)
backrub.set_max_atoms(12)

# Repack side chains in the loop neighborhood without changing residue identity.
task_factory = TaskFactory()
task_factory.push_back(InitializeFromCommandline())
task_factory.push_back(IncludeCurrent())
task_factory.push_back(NoRepackDisulfides())
task_factory.push_back(OperateOnResidueSubset(RestrictToRepackingRLT(), repack_selector, False))
task_factory.push_back(OperateOnResidueSubset(PreventRepackingRLT(), repack_selector, True))

packer = PackRotamersMover()
packer.task_factory(task_factory)
packer.score_function(scorefxn)

# FastRelax can minimize the loop backbone and neighborhood side chains.
relax_movemap = MoveMap()
relax_movemap.set_bb_true_range(FLEX_START, FLEX_END)

for residue in range(1, pose0.total_residue() + 1):
    if repack_subset[residue]:
        relax_movemap.set_chi(residue, True)

# Passing repeats here makes FASTRELAX_REPEATS an active setting.
fast_relax = FastRelax(FASTRELAX_REPEATS)
fast_relax.set_scorefxn(cst_scorefxn)
fast_relax.set_movemap(relax_movemap)
fast_relax.set_task_factory(task_factory)
fast_relax.constrain_relax_to_start_coords(True)
fast_relax.max_iter(FASTRELAX_MAX_ITER)



def save_top_fastrelax_models(results_df, output_directory, number_to_save=5): #save 5 most energetically favorable post-fastrelax models
    top_models_directory = output_directory / "top_5_fastrelax_models"
    top_models_directory.mkdir(parents=True, exist_ok=True)

    # Remove PDBs from an earlier run so this folder contains only the new top 5.
    for old_model in top_models_directory.glob("*.pdb"):
        old_model.unlink()

    # Lower post-FastRelax REU values are treated as more energetically favorable.
    top_models = results_df.nsmallest(number_to_save, "fastrelax_REU").copy()
    top_models.insert(0, "energy_rank", range(1, len(top_models) + 1))

    for _, model in top_models.iterrows():
        source_file = Path(model["final_fastrelax_pdb"])
        destination_file = top_models_directory / source_file.name
        shutil.copy2(source_file, destination_file)

    top_models.to_csv(top_models_directory / "top_5_fastrelax_scores.csv", index=False)
    print(f"Saved the {len(top_models)} most favorable FastRelax models to {top_models_directory}")




results = []

for model_number in range(1, NSTRUCT + 1):
    pose = pose0.clone()
    constraints_added = add_distance_constraints(pose, pose0, FLEX_START, FLEX_END)

    current_score = cst_scorefxn(pose)
    accepted_moves = 0

    for _ in range(BACKRUB_TRIALS):
        old_pose = pose.clone()
        old_score = current_score

        backrub.apply(pose)
        new_score = cst_scorefxn(pose)
        score_change = new_score - old_score

        accept = score_change <= 0
        if not accept:
            accept_probability = math.exp(-score_change / KT)
            accept = random.random() < accept_probability

        if accept:
            current_score = new_score
            accepted_moves += 1
        else:
            pose = old_pose
            current_score = old_score

    packer.apply(pose)

    backrub_repack_score = scorefxn(pose)
    backrub_repack_cst_score = cst_scorefxn(pose)

    pre_relax_file = OUTDIR / f"5goa_A165D_open_backrub_repack_before_fastrelax_{model_number:02d}.pdb" #CHANGE NAMING
    pose.dump_pdb(str(pre_relax_file))

    fast_relax.apply(pose)

    final_score = scorefxn(pose)
    final_cst_score = cst_scorefxn(pose)
    delta = final_score - original_score

    final_file = OUTDIR / f"5goa_A165D_open_backrub_fastrelax_{model_number:02d}.pdb" #CHANGE NAMING
    pose.dump_pdb(str(final_file))

    print(
        f"Model {model_number:02d} | "
        f"Backrub + repack: {backrub_repack_score:.3f} REU | "
        f"FastRelax: {final_score:.3f} REU | "
        f"Delta: {delta:.3f} REU | "
        f"Accepted: {accepted_moves}/{BACKRUB_TRIALS} | "
        f"Constraints: {constraints_added}"
    )

    results.append(
        {
            "model_number": model_number,
            "backrub_repack_REU": backrub_repack_score,
            "backrub_repack_constraint_REU": backrub_repack_cst_score,
            "fastrelax_REU": final_score,
            "fastrelax_constraint_REU": final_cst_score,
            "delta_from_original_REU": delta,
            "accepted_moves": accepted_moves,
            "backrub_trials": BACKRUB_TRIALS,
            "constraints_added": constraints_added,
            "pre_fastrelax_pdb": str(pre_relax_file),
            "final_fastrelax_pdb": str(final_file),
        }
    )


df = pd.DataFrame(results)
csv_file = OUTDIR / "A165D_open_fastrelax_scores.csv" #CHANGE NAME
df.to_csv(csv_file, index=False)

# Rank all completed models and copy only the five lowest-energy final PDBs.
save_top_fastrelax_models(df, OUTDIR, number_to_save=5)

display(df)

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.Release.python312.ubuntu 2025.37+release.df75a9c48e763e52a7aa3f5dfba077f4da88dbf5 2025-09-03T12:23:30] retrieved from: http://www.pyrosetta.org
Original score: 519.511 REU
Model 01 | Backrub + repack: 551.310 REU | FastRelax: 504.787 REU | Delta: -14.724 REU | Accepted: 181/250 | Constraints: 1701
Model 02

,model_number,backrub_repack_REU,backrub_repack_constraint_REU,fastrelax_REU,fastrelax_constraint_REU,delta_from_original_REU,accepted_moves,backrub_trials,constraints_added,pre_fastrelax_pdb,final_fastrelax_pdb
0,1,551.309571,551.409942,504.786989,504.882439,-14.724267,181,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...
1,2,590.849400,591.098461,512.547381,513.148313,-6.963876,184,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...
2,3,568.667069,568.881177,505.349043,505.447407,-14.162213,156,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...
3,4,574.532220,574.594705,502.052599,502.099082,-17.458658,161,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...
4,5,583.628505,583.758731,512.897648,513.072388,-6.613608,171,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...
5,6,551.239057,551.297276,498.994824,499.080725,-20.516432,162,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...
6,7,565.024923,565.112168,503.794419,504.092611,-15.716837,164,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...
7,8,553.009846,553.142746,496.108132,496.224171,-23.403125,172,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...
8,9,569.507668,569.621699,509.025540,509.107912,-10.485716,180,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...
9,10,568.606646,568.691193,506.463873,506.556880,-13.047384,170,250,1701,/mnt/c/Users/kevin/Downloads/A165D open fastre...,/mnt/c/Users/kevin/Downloads/A165D open fastre...


In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyrosetta import pose_from_pdb
from pyrosetta.io import pose_from_pdbstring


#Change only these three for desired results

# INSERT the top-5 folder created by the previous FastRelax cell here.
MODELS_FOLDER = Path(
    "/mnt/c/Users/kevin/Downloads/A165D open fastrelax/top_5_fastrelax_models"
)

# INSERT the base PDB that all five FastRelax models should be compared against.
BASE_PDB = Path(
    "/mnt/c/Users/kevin/Downloads/RYR2_PDB_STRUCTURES/PDB_STRUCTURES/5goa_A165D.pdb"
)

# INSERT the folder where the CSV files and heatmaps should be saved.
OUTPUT_FOLDER = Path(
    "/mnt/c/Users/kevin/Downloads/A165D open fastrelax/top_5_distance_analysis"
)


OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Automatically collect all five post-FastRelax PDBs from the previous cell.
pdb_files = sorted(MODELS_FOLDER.glob("*.pdb"))
print(f"Found {len(pdb_files)} post-FastRelax models.")

CHAIN = "A"
RES_RANGE = range(160, 181)

CA_VMAX = 1.0
ORIENT_VMAX = 25.0


def ca_xyz(residue):
    """Return the C-alpha coordinates of a residue."""
    return residue.xyz(residue.atom_index("CA"))


def sidechain_orientation_vector(residue):
    """Return the vector from the C-alpha atom to the side-chain centroid."""
    ca = residue.xyz(residue.atom_index("CA"))
    sidechain_coordinates = []

    for atom_number in range(1, residue.nheavyatoms() + 1):
        atom_name = residue.atom_name(atom_number).strip()
        if atom_name in ["N", "CA", "C", "O"]:
            continue

        xyz = residue.xyz(atom_number)
        sidechain_coordinates.append([xyz.x, xyz.y, xyz.z])

    if len(sidechain_coordinates) == 0:
        return np.array([0.0, 0.0, 0.0])

    centroid = np.mean(sidechain_coordinates, axis=0)
    return centroid - np.array([ca.x, ca.y, ca.z])


def angle_between(vector1, vector2):
    """Return the angle in degrees between two vectors."""
    norm1 = np.linalg.norm(vector1)
    norm2 = np.linalg.norm(vector2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    cosine = np.clip(np.dot(vector1, vector2) / (norm1 * norm2), -1.0, 1.0)
    return np.degrees(np.arccos(cosine))


def collect_residues(pose, chain, residue_range):
    """Collect residues by their PDB residue numbers."""
    residues = {}

    for pose_number in range(1, pose.size() + 1):
        pdb_number = pose.pdb_info().number(pose_number)
        pdb_chain = pose.pdb_info().chain(pose_number)

        if pdb_chain == chain and pdb_number in residue_range:
            residues[pdb_number] = pose.residue(pose_number)

    return residues


# Load the selected base structure once. Every model is compared with this pose.
base_pose = pose_from_pdb(str(BASE_PDB))
base_residues = collect_residues(base_pose, CHAIN, RES_RANGE)


for pdb_file in pdb_files:
    basename = pdb_file.stem
    print(f"\nProcessing {basename}...")

    # Read with Python first, then pass the PDB text to Rosetta. This avoids
    # Rosetta's file-opening failure for the copied Windows/WSL path.
    pdb_contents = pdb_file.read_text()
    model_pose = pose_from_pdbstring(pdb_contents)
    model_residues = collect_residues(model_pose, CHAIN, RES_RANGE)

    ca_shifts = []
    orientation_changes = []
    residue_labels = []

    for residue_number in RES_RANGE:
        base_residue = base_residues[residue_number]
        model_residue = model_residues[residue_number]

        ca_shift = ca_xyz(model_residue).distance(ca_xyz(base_residue))
        ca_shifts.append(ca_shift)

        orientation_change = angle_between(
            sidechain_orientation_vector(base_residue),
            sidechain_orientation_vector(model_residue),
        )
        orientation_changes.append(orientation_change)

        if base_residue.name1() != model_residue.name1():
            label = f"{base_residue.name1()}{residue_number}{model_residue.name1()}"
        else:
            label = f"{model_residue.name1()}{residue_number}"
        residue_labels.append(label)

    results = pd.DataFrame(
        {
            "Residue": residue_labels,
            "Cα_shift": ca_shifts,
            "Sidechain_orientation_change_deg": orientation_changes,
        }
    )

    csv_path = OUTPUT_FOLDER / f"{basename}_base_relative.csv"
    results.to_csv(csv_path, index=False)
    print(f"Saved CSV: {csv_path}")

    # C-alpha displacement heatmap for this model relative to the selected base.
    fig, ax = plt.subplots(figsize=(6, 10))
    sns.heatmap(
        np.array(ca_shifts)[:, np.newaxis],
        ax=ax,
        cmap="mako",
        vmin=0,
        vmax=CA_VMAX,
        cbar_kws={"label": "Cα shift (Å)"},
        yticklabels=residue_labels,
        xticklabels=["Cα shift (Å)"],
        annot=True,
        fmt=".2f",
        annot_kws={"fontsize": 14},
    )
    ax.set_title(f"{basename} - base-relative Cα shift", fontsize=14)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
    ax.set_ylabel("NT-A residue number", fontsize=12)
    plt.tight_layout()

    ca_figure_path = OUTPUT_FOLDER / f"{basename}_Calpha_shift.png"
    plt.savefig(ca_figure_path, dpi=300)
    plt.close()
    print(f"Saved Cα heatmap: {ca_figure_path}")

    # Side-chain orientation heatmap for this model relative to the selected base.
    fig, ax = plt.subplots(figsize=(6, 10))
    sns.heatmap(
        np.array(orientation_changes)[:, np.newaxis],
        ax=ax,
        cmap="rocket",
        vmin=0,
        vmax=ORIENT_VMAX,
        cbar_kws={"label": "Side-chain orientation change (deg)"},
        yticklabels=residue_labels,
        xticklabels=["ΔOrientation"],
        annot=True,
        fmt=".2f",
        annot_kws={"fontsize": 14},
    )
    ax.set_title(f"{basename} - base-relative ΔSide-chain orientation", fontsize=14)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
    ax.set_ylabel("NT-A residue number", fontsize=12)
    plt.tight_layout()

    orientation_figure_path = OUTPUT_FOLDER / f"{basename}_Sidechain_orientation.png"
    plt.savefig(orientation_figure_path, dpi=300)
    plt.close()
    print(f"Saved side-chain orientation heatmap: {orientation_figure_path}")

print("\nFinished comparing every selected FastRelax model with the same base PDB.")


Found 5 post-FastRelax models.

Processing 5goa_A165D_open_backrub_fastrelax_04...
Saved CSV: /mnt/c/Users/kevin/Downloads/A165D open fastrelax/top_5_distance_analysis/5goa_A165D_open_backrub_fastrelax_04_base_relative.csv
Saved Cα heatmap: /mnt/c/Users/kevin/Downloads/A165D open fastrelax/top_5_distance_analysis/5goa_A165D_open_backrub_fastrelax_04_Calpha_shift.png
Saved side-chain orientation heatmap: /mnt/c/Users/kevin/Downloads/A165D open fastrelax/top_5_distance_analysis/5goa_A165D_open_backrub_fastrelax_04_Sidechain_orientation.png

Processing 5goa_A165D_open_backrub_fastrelax_06...
Saved CSV: /mnt/c/Users/kevin/Downloads/A165D open fastrelax/top_5_distance_analysis/5goa_A165D_open_backrub_fastrelax_06_base_relative.csv
Saved Cα heatmap: /mnt/c/Users/kevin/Downloads/A165D open fastrelax/top_5_distance_analysis/5goa_A165D_open_backrub_fastrelax_06_Calpha_shift.png
Saved side-chain orientation heatmap: /mnt/c/Users/kevin/Downloads/A165D open fastrelax/top_5_distance_analysis/5goa_A